# Battery arbitrage valuation — corrected

Same business case, with the modelling and unit errors fixed. Each fix is marked **Fix N**
(numbers refer to `mock_11_solution.md`).

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

pd.set_option("display.width", 120)

## Load data

In [2]:
df = pd.read_csv("../../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
prices = df["price_eur_mwh"]
step = prices.index.to_series().diff().value_counts()
print(step.head(3))
prices.describe().round(1)

time
0 days 01:00:00    17519
Name: count, dtype: int64


count    17520.0
mean        98.5
std         36.9
min        -19.9
25%         73.5
50%         97.7
75%        122.8
max        419.6
Name: price_eur_mwh, dtype: float64

**Fix 4 (units).** The data is hourly, not half-hourly: one row per hour, 24 per day. A 50 MW
limit is therefore 50 MWh per period.

**Fix 5 (negative prices).** Negative prices are kept. A battery is *paid* to charge in those
hours, they are among the most valuable of the year, and dropping the rows also left those days
with 23 periods so they were silently skipped by the `try/except` in the loop.

In [3]:
E_MAX = 100.0
P_MAX = 50.0           # MW * 1h = MWh per hourly period
ETA = 0.90
SOC0 = 50.0
FX = 0.87

print("negative-price hours:", (prices < 0).sum(),
      "on", prices[prices < 0].index.normalize().nunique(), "days")

negative-price hours: 43 on 38 days


## Daily LP

**Fix 1 (objective sign).** `linprog` minimises. To maximise revenue `sum(p*(dis - ch))` we
minimise its negative, so the cost vector is `[+p for charge, -p for discharge]`. The original
`[-p, +p]` *minimised* revenue: the LP charged at the peak and discharged at the trough, and the
`abs()` later hid the sign.

**Fix 2 (efficiency side).** Energy stored per MWh drawn from the grid is `eta * charge`, not
`charge / eta`. Dividing made the battery create energy.

**Fix 3 (terminal state).** Without an end-of-day constraint the optimiser sells the initial
50 MWh every day for free. Require `soc_24 == SOC0` so each day is self-financing.

**Fix 8 (degradation in the objective).** With a marginal cycle cost of £1,500 per 100 MWh
(about €17/MWh discharged) the optimiser should only cycle when the spread pays for it.
Charging it after the fact lets the LP chase €5 wiggles and then bills them as degradation.

**Fix 12 (status).** Check `res.status` and count rather than swallow failures.

In [4]:
CYCLE_COST = 1500.0                                   # GBP per equivalent full cycle
DEG_EUR_MWH = CYCLE_COST / FX / E_MAX                 # EUR per MWh discharged
L = np.tril(np.ones((24, 24)))

def schedule(p):
    c = np.concatenate([p, -p + DEG_EUR_MWH])         # minimise -(revenue - degradation)
    A_soc = np.hstack([L * ETA, -L])                  # soc_t - SOC0 = sum(eta*ch - dis)
    A_ub = np.vstack([A_soc, -A_soc])
    b_ub = np.concatenate([np.full(24, E_MAX - SOC0), np.full(24, SOC0)])
    A_eq = A_soc[-1:]                                 # end the day where it started
    b_eq = np.array([0.0])
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                  bounds=[(0, P_MAX)] * 48, method="highs")
    if res.status != 0:
        raise RuntimeError(res.message)
    return res.x[:24], res.x[24:]

In [5]:
def run(price_for_schedule, price_realised):
    rows, failed = [], []
    days = sorted(set(price_realised.index.date))
    for day in days:
        ps = price_for_schedule.get(day)
        if ps is None or len(ps) != 24:
            failed.append(day); continue
        pa = price_realised.loc[str(day)].values
        try:
            ch, dis = schedule(ps)
        except RuntimeError as e:
            failed.append(day); continue
        rows.append({"day": pd.Timestamp(day),
                     "revenue_eur": float((pa * (dis - ch)).sum()),
                     "charged": ch.sum(), "discharged": dis.sum(),
                     "simultaneous_h": int(((ch > 1e-6) & (dis > 1e-6)).sum())})
    return pd.DataFrame(rows).set_index("day"), failed

by_day = {d: g.values for d, g in prices.groupby(prices.index.date)}
daily, failed = run(by_day, prices)
print(len(daily), "days scheduled,", len(failed), "failed")
daily.describe().round(1)

730 days scheduled, 0 failed


,revenue_eur,charged,discharged,simultaneous_h
count,730.0,730.0,730.0,730.0
mean,9378.1,199.9,179.9,0.0
std,3079.8,49.0,44.1,0.0
min,2461.5,105.6,95.0,0.0
25%,7335.7,161.1,145.0,0.0
50%,8962.8,211.1,190.0,0.0
75%,10671.2,222.2,200.0,0.0
max,24421.0,333.3,300.0,0.0


**Fix 7 (foresight).** Scheduling against the realised day-ahead prices is *perfect foresight*.
It is an upper bound, not a "realistic" number. A minimal realistic variant schedules on the
average hourly price profile of the previous seven days and is settled at the actual prices;
a real desk would use a proper price forecast and sit somewhere between the two.

In [6]:
days = sorted(by_day)
naive_sched = {days[i]: np.mean([by_day[days[j]] for j in range(i - 7, i)], axis=0)
               for i in range(7, len(days))}
daily_naive, failed_naive = run(naive_sched, prices)
print(len(daily_naive), "days,", len(failed_naive), "failed")

723 days, 7 failed


## Sanity check on one day: the schedule should charge cheap and discharge dear

In [7]:
day = "2022-12-12"
p = prices.loc[day].values
ch, dis = schedule(p)
pd.DataFrame({"price": p, "charge": ch.round(1), "discharge": dis.round(1)}, index=range(24)).T

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
price,106.24,99.31,83.86,72.69,89.98,103.12,89.73,136.03,138.34,121.91,...,118.1,131.41,134.71,174.2,168.63,153.85,139.32,139.08,136.06,105.27
charge,0.00,0.00,5.60,50.00,0.00,0.00,0.00,0.00,0.00,0.00,...,50.0,-0.00,-0.00,0.0,0.00,0.00,0.00,0.00,5.60,50.00
discharge,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0.00,0.00,50.0,50.00,0.00,0.00,0.00,0.00,0.00


In [8]:
print("hours with simultaneous charge and discharge:", daily["simultaneous_h"].sum())
daily.loc[daily["simultaneous_h"] > 0].head()

hours with simultaneous charge and discharge: 0


,revenue_eur,charged,discharged,simultaneous_h
day,,,,


**Fix 6 (FX applied once).** Convert once, at the daily level, and never again.

**Fix 8 (cycles).** One equivalent full cycle is `E_MAX` MWh *discharged* (throughput on one
side). Dividing by `2 * E_MAX` halved the degradation charge.

In [9]:
for d in (daily, daily_naive):
    d["revenue_gbp"] = d["revenue_eur"] * FX
    d["cycles"] = d["discharged"] / E_MAX
    d["net_gbp"] = d["revenue_gbp"] - d["cycles"] * CYCLE_COST

by_year = pd.DataFrame({
    "perfect foresight net £m": daily["net_gbp"].groupby(daily.index.year).sum() / 1e6,
    "naive (7-day profile) net £m": daily_naive["net_gbp"].groupby(daily_naive.index.year).sum() / 1e6,
    "cycles / yr": daily["cycles"].groupby(daily.index.year).sum(),
}).round(2)
by_year

,perfect foresight net £m,naive (7-day profile) net £m,cycles / yr
day,,,
2022,1.88,0.71,620.7
2023,2.11,0.87,692.4


**Fix 9 (annualisation).** Show both years; 2022 was a gas-spike regime. Use the average, or
better, the more conservative year, and say which.

**Fix 10 (capex units).** £300/kWh is a price per unit of *energy* capacity: multiply by
100,000 kWh, not by the 50,000 kW power rating.

In [10]:
capex_gbp = 300 * (E_MAX * 1000)
annual_pf = daily["net_gbp"].sum() / 2
annual_naive = daily_naive["net_gbp"].sum() / (len(daily_naive) / 365)
summary = pd.DataFrame({
    "annual net (£m/yr)": [annual_pf / 1e6, annual_naive / 1e6],
    "capex (£m)": [capex_gbp / 1e6] * 2,
    "payback (years)": [capex_gbp / annual_pf, capex_gbp / annual_naive],
}, index=["perfect foresight (upper bound)", "naive schedule"]).round(2)
summary

,annual net (£m/yr),capex (£m),payback (years)
perfect foresight (upper bound),1.99,30.0,15.05
naive schedule,0.80,30.0,37.52


## Results

In [11]:
print(f"days scheduled: {len(daily)} (failed: {len(failed)})")
print(f"perfect-foresight annual net: £{annual_pf/1e6:.2f}m  -> payback {capex_gbp/annual_pf:.1f} y (upper bound)")
print(f"naive-schedule annual net:    £{annual_naive/1e6:.2f}m  -> payback {capex_gbp/annual_naive:.1f} y")
print(f"cycles per year: {daily['cycles'].sum()/2:.0f}")

days scheduled: 730 (failed: 0)
perfect-foresight annual net: £1.99m  -> payback 15.1 y (upper bound)
naive-schedule annual net:    £0.80m  -> payback 37.5 y
cycles per year: 657


The honest range is bounded above by perfect foresight and below by a naive profile schedule;
the achievable number depends on price-forecast quality, which this notebook does not model.
Even the upper bound pays back only within the battery's life, not comfortably inside it, and the
600-700 cycles a year sit near typical warranty limits. "Build five more" is not supported.